In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
save_directory = '/content/drive/MyDrive/Project/ArSarcasm-v2/03_train/03'

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, BertPreTrainedModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score)

from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter

from transformers import AutoTokenizer, AutoModel

import torch.nn.functional as F

from sklearn.metrics import precision_recall_curve

from typing import Dict, List, Tuple

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, texts, labels, sources, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.sources = sources  # Added source information
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        source = self.sources[idx]  # Get source info

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
            'source': torch.tensor(1 if source == 'original' else 0, dtype=torch.float)  # Binary source indicator
        }

In [ ]:
class MultiTaskBert(BertPreTrainedModel):
    def __init__(self, config, num_labels_per_task, model_name, dropout_probability):
        super().__init__(config)
        self.bert = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = nn.Dropout(p=dropout_probability)

        self.classification_heads = nn.ModuleDict({
            f'task_{i}': nn.Linear(config.hidden_size, num_labels)
            for i, num_labels in enumerate(num_labels_per_task)
        })

    def forward(self, input_ids, attention_mask, task=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        pooled_output = torch.mean(hidden_states, dim=1)
        pooled_output = self.dropout(pooled_output)
        return self.classification_heads[task](pooled_output)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, original_weight=1.0, generated_weight=0.7, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.original_weight = original_weight
        self.generated_weight = generated_weight

    def forward(self, inputs, targets, sources=None):
        logpt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(logpt)

        logpt = logpt.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = pt.gather(1, targets.unsqueeze(1)).squeeze(1)

        loss = - (1 - pt) ** self.gamma * logpt

        if self.alpha is not None:
            # Move alpha to the same device as targets
            alpha = self.alpha.to(targets.device)
            alpha_factor = alpha.gather(0, targets)
            loss = alpha_factor * loss

        # Apply source-based weighting if sources are provided
        if sources is not None:
            source_weights = torch.where(
                sources == 1,  # 1 for original, 0 for generated
                torch.tensor(self.original_weight, device=targets.device),
                torch.tensor(self.generated_weight, device=targets.device)
            )
            loss = loss * source_weights

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [ ]:
def create_data_loader(df, tokenizer, max_length, batch_size):
    ds = CustomDataset(
        texts=df.text.to_numpy(),
        labels=df.label.to_numpy(),
        sources=df.source.to_numpy() if 'source' in df.columns else np.array(['original'] * len(df)),  # Add source info
        tokenizer=tokenizer,
        max_length=max_length
    )
    return DataLoader(ds, batch_size=batch_size, num_workers=2)

In [ ]:
def compute_metrics(true_labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(true_labels, preds)

    report = classification_report(true_labels, preds, zero_division=0)
    print(report)

    return precision, recall, f1, acc

In [ ]:
def evaluate_model(model, val_data_loaders, loss_functions, label_encoders):  # Added label_encoders parameter
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    task_results = {
        task_id: {"texts": [], "preds": [], "labels": [], "losses": [], "probs": []}
        for task_id in range(len(val_data_loaders))
    }

    with torch.no_grad():
        for task_id, val_loader in enumerate(val_data_loaders):
            if len(val_loader) == 0:  # Skip empty loaders
                continue

            # Process all batches for this task
            for batch in val_loader:
                texts = batch["text"]
                task_results[task_id]["texts"].extend(texts)

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask, task=f'task_{task_id}')

                if "source" in batch:
                    sources = batch["source"].to(device)
                else:
                    sources = torch.ones_like(labels).float().to(device)
                loss = loss_functions[task_id](logits, labels, sources)

                probs = torch.softmax(logits, dim=1).cpu().numpy()
                preds = logits.argmax(1)

                # Store batch results
                task_results[task_id]["preds"].extend(preds.cpu().numpy())
                task_results[task_id]["labels"].extend(labels.cpu().numpy())
                task_results[task_id]["losses"].append(loss.item())
                task_results[task_id]["probs"].append(probs)

            # Print confidence stats AFTER processing all batches for this task
            if task_results[task_id]["probs"]:
                all_probs = np.concatenate(task_results[task_id]["probs"], axis=0)
                classes = label_encoders[task_id].classes_
                print(f"\nTask {task_id} Confidence Stats:")
                for class_idx, class_name in enumerate(classes):
                    class_probs = all_probs[:, class_idx]
                    print(f"Val Class {class_idx} ({class_name}): "
                          f"Mean={class_probs.mean():.2f}, "
                          f"Max={class_probs.max():.2f}")

    # Compute metrics per task
    metrics = {}
    total_f1 = 0.0
    total_loss = 0.0
    task_sample_counts = []

    for task_id in task_results:
        if len(task_results[task_id]["losses"]) == 0:
            continue

        labels = np.array(task_results[task_id]["labels"])
        preds = np.array(task_results[task_id]["preds"])
        losses = task_results[task_id]["losses"]
        probs = np.concatenate(task_results[task_id]["probs"], axis=0)

        print(f'\n\nEvaluation metrics for Task {task_id}:')
        precision, recall, f1, acc = compute_metrics(labels, preds)
        avg_loss = np.mean(losses)

        metrics[task_id] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": acc,
            "avg_loss": avg_loss,
            "n_samples": len(labels),
            "labels": labels,
            "preds": preds,
            "probs": probs
        }
        total_f1 += f1
        total_loss += avg_loss
        task_sample_counts.append(len(labels))

    # Calculate aggregate metrics
    macro_f1 = total_f1 / len(metrics) if metrics else 0
    avg_loss = total_loss / len(metrics) if metrics else 0
    weighted_f1 = sum(metrics[t]["f1"] * metrics[t]["n_samples"] for t in metrics) / sum(task_sample_counts) if metrics else 0

    print(f'\nValidation Completed - Tasks Processed: {len(metrics)}')
    return metrics, macro_f1, weighted_f1, avg_loss, task_results

In [ ]:
def calculate_class_metrics_and_confusion_matrix(all_labels, all_preds, label_encoder):
    all_labels_transformed = label_encoder.inverse_transform(all_labels)
    all_preds_transformed = label_encoder.inverse_transform(all_preds)

    class_report = classification_report(all_labels_transformed, all_preds_transformed, output_dict=True)
    conf_matrix = confusion_matrix(all_labels_transformed, all_preds_transformed)
    return class_report, conf_matrix

In [ ]:
def print_dialect_decomposition_table(test_results, label_encoders, task_id=2):
    """
    Prints per-class Precision, Recall, F1 and support for the dialect task.
    """
    labels = np.array(test_results[task_id]["labels"])
    preds = np.array(test_results[task_id]["preds"])
    classes = label_encoders[task_id].classes_

    p, r, f1, support = precision_recall_fscore_support(
        labels, preds, labels=np.arange(len(classes)), zero_division=0
    )

    print(f"\n{'='*70}")
    print(f"DIALECT TASK (Task {task_id}) — Per-Class Decomposition")
    print(f"{'='*70}")
    print(f"{'Class':<<12} {'Support':>8} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
    print("-" * 70)

    for idx, cls in enumerate(classes):
        print(f"{cls:<12} {support[idx]:>8} {p[idx]:>10.4f} {r[idx]:>10.4f} {f1[idx]:>10.4f}")

    print("-" * 70)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    print(f"{'MACRO AVG':<<12} {'':>8} {macro_p:>10.4f} {macro_r:>10.4f} {macro_f1:>10.4f}")
    print(f"{'='*70}\n")


def bootstrap_task_f1(test_results, label_encoders, save_directory, task_id=2,
                      n_bootstrap=10000, random_state=42):
    rng = np.random.RandomState(random_state)
    labels = np.array(test_results[task_id]["labels"])
    preds = np.array(test_results[task_id]["preds"])
    n_samples = len(labels)
    classes = label_encoders[task_id].classes_

    macro_f1_all = []
    macro_f1_no_magreb = []
    per_class_f1 = {cls: [] for cls in classes}

    magreb_idx = list(classes).index('magreb') if 'magreb' in classes else None

    for _ in range(n_bootstrap):
        indices = rng.randint(0, n_samples, size=n_samples)
        b_labels = labels[indices]
        b_preds = preds[indices]

        p, r, f1, _ = precision_recall_fscore_support(
            b_labels, b_preds, labels=np.arange(len(classes)), zero_division=0
        )

        macro_f1_all.append(np.mean(f1))

        if magreb_idx is not None:
            mask = np.ones(len(classes), dtype=bool)
            mask[magreb_idx] = False
            macro_f1_no_magreb.append(np.mean(f1[mask]))

        for idx, cls in enumerate(classes):
            per_class_f1[cls].append(f1[idx])

    def ci(scores):
        return np.percentile(scores, [2.5, 97.5])

    lines = []
    lines.append("="*70)
    lines.append(f"BOOTSTRAP ANALYSIS REPORT (n={n_bootstrap}) — Task {task_id}")
    lines.append("="*70)
    lines.append(f"\nMacro F1 (all classes):     Mean={np.mean(macro_f1_all):.4f}  95% CI={ci(macro_f1_all)}")
    if magreb_idx is not None:
        lines.append(f"Macro F1 (excl. magreb):  Mean={np.mean(macro_f1_no_magreb):.4f}  95% CI={ci(macro_f1_no_magreb)}")

    lines.append("\nPer-class F1 95% Confidence Intervals:")
    for cls in classes:
        mean_f1 = np.mean(per_class_f1[cls])
        lower, upper = ci(per_class_f1[cls])
        lines.append(f"  {cls:12s}: Mean={mean_f1:.4f}  95% CI=[{lower:.4f}, {upper:.4f}]")
    lines.append("="*70)

    report = "\n".join(lines)
    print(report)

    report_path = os.path.join(save_directory, f"bootstrap_report_task{task_id}.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report)
    print(f"\n[SAVED] Bootstrap report -> {report_path}\n")

    return {
        'macro_f1_all_mean': np.mean(macro_f1_all),
        'macro_f1_all_ci': ci(macro_f1_all),
        'macro_f1_no_magreb_mean': np.mean(macro_f1_no_magreb) if magreb_idx else None,
        'macro_f1_no_magreb_ci': ci(macro_f1_no_magreb) if magreb_idx else None,
    }

In [ ]:
def calculate_test_model_all_tasks_and_show_results(model_config, model, tokenizer, df_test,
                                                  label_encoders, loss_functions, best_thresholds=None):
    MAX_NUMBER_OF_TOKENS = model.config.max_position_embeddings
    task_id_to_name = model_config['task_id_to_name']
    batch_size = model_config['batch_size']

    # Create data loaders
    test_data_loaders = [create_data_loader(df, tokenizer, MAX_NUMBER_OF_TOKENS, batch_size)
                         for df in df_test]

    # Evaluate model with label_encoders passed
    test_metrics, test_macro_f1, test_weighted_f1, test_avg_loss, test_results = evaluate_model(
        model, test_data_loaders, loss_functions, label_encoders  # Added label_encoders
    )

    print_dialect_decomposition_table(test_results, label_encoders, task_id=2)
    bootstrap_task_f1(
        test_results=test_results,
        label_encoders=label_encoders,
        save_directory=model_config['save_directory'],
        task_id=2,
        n_bootstrap=10000
    )


    results_dfs = {}

    for task_id, metrics in test_metrics.items():
        task_name = task_id_to_name.get(task_id, f"Task {task_id}")
        probs = np.concatenate(test_results[task_id]["probs"], axis=0)
        texts = test_results[task_id]["texts"]
        true_labels = np.array(test_results[task_id]["labels"])
        classes = label_encoders[task_id].classes_

        # Threshold handling
        if best_thresholds and task_id in best_thresholds:
            task_thresholds = best_thresholds[task_id]
            thresholds = np.array([task_thresholds[cls] for cls in classes])
            above_or_equals_threshold = probs >= thresholds

            final_preds = np.zeros(probs.shape[0], dtype=int)
            for i in range(probs.shape[0]):
                valid_indices = np.where(above_or_equals_threshold[i])[0]
                if valid_indices.size > 0:
                    final_preds[i] = valid_indices[np.argmax(probs[i, valid_indices])]
                else:
                    final_preds[i] = np.argmax(probs[i])
        else:
            final_preds = np.argmax(probs, axis=1)

        # Create and store results
        task_df = create_task_dataframe(
            texts, true_labels, final_preds, label_encoders[task_id], probs
        )
        results_dfs[task_name] = task_df

    return results_dfs

In [ ]:
def create_task_dataframe(texts, true_labels, preds, label_encoder, probs):
    df = pd.DataFrame({
        "text": texts,
        "true_label": label_encoder.inverse_transform(true_labels),
        "predicted_label": label_encoder.inverse_transform(preds)
    })
    for idx, class_name in enumerate(label_encoder.classes_):
        df[f'prob_{class_name}'] = probs[:, idx]
    return df

In [ ]:
import os

def run_test_with_thresholds(model, tokenizer, test_df, label_encoders,
                            loss_functions, train_config, suffix, thresholds=None):
    results = calculate_test_model_all_tasks_and_show_results(
        train_config, model, tokenizer, test_df,
        label_encoders, loss_functions, thresholds
    )

    # Create save directory if missing
    save_dir = train_config['save_directory']
    os.makedirs(save_dir, exist_ok=True)

    return results

In [ ]:
def get_task_thresholds(val_results, label_encoders):
    best_thresholds = {}
    for task_id in range(len(val_results)):
        classes = label_encoders[task_id].classes_
        val_probs = np.concatenate(val_results[task_id]["probs"])
        val_true = np.array(val_results[task_id]["labels"])

        task_thresholds = {}
        for class_idx, class_name in enumerate(classes):
            y_true = (val_true == class_idx).astype(int)
            if np.sum(y_true) == 0:
                task_thresholds[class_name] = 1.0
                continue

            y_scores = val_probs[:, class_idx]
            precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
            thresholds = np.append(thresholds, 1.0)
            f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)
            best_idx = np.nanargmax(f1_scores)
            task_thresholds[class_name] = thresholds[best_idx]

        best_thresholds[task_id] = task_thresholds
    return best_thresholds

In [ ]:
def load_trained_model(train_config, model_file):
    dir = train_config['save_directory']
    model = torch.load(f'{dir}/{model_file}', weights_only=False)
    tokenizer = AutoTokenizer.from_pretrained(train_config['model_name'])
    return model, tokenizer

In [ ]:
train_csv = '/content/drive/MyDrive/Project/ArSarcasm-v2/01_data_preparation/train.csv'
val_csv = '/content/drive/MyDrive/Project/ArSarcasm-v2/01_data_preparation/val.csv'
test_csv = '/content/drive/MyDrive/Project/ArSarcasm-v2/01_data_preparation/test.csv'
generated_csv = '/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/generated.csv'

In [ ]:
original_train_df = pd.read_csv(train_csv)
original_test_df = pd.read_csv(test_csv)
generated_df = pd.read_csv(generated_csv)

original_train_df['source'] = 'original'
generated_df['source'] = 'generated'

original_train_sampled = original_train_df.groupby('combined_label').apply(
    lambda x: x.sample(min(len(x), 1000), random_state=42)
).reset_index(drop=True)

all_train_df = pd.concat([original_train_sampled.copy(), generated_df.copy()]).sample(frac=1, random_state=42).reset_index(drop=True)
all_test_df = original_test_df.copy().sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
sarcasm_train = all_train_df.copy()[['text', 'sarcasm', 'source']].rename(columns={'sarcasm': 'label'})
sarcasm_train['label'] = sarcasm_train['label'].astype(str)  # Convert to string

sentiment_train = all_train_df.copy()[['text', 'sentiment', 'source']].rename(columns={'sentiment': 'label'})
sentiment_train['label'] = sentiment_train['label'].astype(str)  # Convert to string

dialect_train = all_train_df.copy()[['text', 'dialect', 'source']].rename(columns={'dialect': 'label'})
dialect_train['label'] = dialect_train['label'].astype(str)  # Convert to string

sarcasm_test = all_test_df.copy()[['text', 'sarcasm']].rename(columns={'sarcasm': 'label'})
sarcasm_test['label'] = sarcasm_test['label'].astype(str)

sentiment_test = all_test_df.copy()[['text', 'sentiment']].rename(columns={'sentiment': 'label'})
sentiment_test['label'] = sentiment_test['label'].astype(str)

dialect_test = all_test_df.copy()[['text', 'dialect']].rename(columns={'dialect': 'label'})
dialect_test['label'] = dialect_test['label'].astype(str)

train_df = [sarcasm_train, sentiment_train, dialect_train]
test_df = [sarcasm_test, sentiment_test, dialect_test]

In [ ]:
train_config = {
    "model_name": 'aubmindlab/bert-base-arabertv02',
    "task_id_to_name": { 0: "Sarcasm", 1: "Sentiment", 2: "Dialect" },
    "batch_size": 32,
    "save_directory": save_directory,
    "learning_rate": 5e-5,
    "dropout_probability": 0.3,
}

In [ ]:
# After label encoding:
label_encoders = [LabelEncoder().fit(df.label) for df in train_df]

for i, df in enumerate(train_df):
    df['label'] = label_encoders[i].transform(df['label'])
for i, df in enumerate(test_df):
    df['label'] = label_encoders[i].transform(df['label'])

num_labels_per_task = [df.label.nunique() for df in train_df]

print(num_labels_per_task)

In [ ]:
def get_task_class_weights(train_dfs, label_encoders):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    task_alphas = []

    for task_id, (df, le) in enumerate(zip(train_dfs, label_encoders)):
        # Get class counts for this task
        class_counts = df['label'].value_counts().sort_index()
        classes = le.classes_

        # Calculate weights using inverse frequency
        total = class_counts.sum()
        weights = torch.tensor([
            total / class_counts[cls_idx]
            for cls_idx in range(len(classes))
        ], dtype=torch.float32)

        # Normalize weights
        weights /= weights.sum()

        task_alphas.append(weights.to(device))

        print(f"Task {task_id} ({le.classes_}) weights: {weights.tolist()}")

    return task_alphas

In [ ]:
task_alphas = get_task_class_weights(train_df, label_encoders)

loss_functions = [
    FocalLoss(alpha=alpha, gamma=2, reduction='mean')
    for alpha in task_alphas
]

print(f'task_alphas={task_alphas}')
print(f'loss_functions={loss_functions}')

In [ ]:
def load_trained_model(train_config, model_file):
    dir = train_config['save_directory']
    model = torch.load(f'{dir}/{model_file}', weights_only=False)
    tokenizer = AutoTokenizer.from_pretrained(train_config['model_name'])
    return model, tokenizer

In [ ]:
model, tokenizer = load_trained_model(train_config, 'best_multi_task_model.bin')

task_names = [
    train_config["task_id_to_name"][task_id]
    for task_id in sorted(train_config["task_id_to_name"].keys())
]

In [ ]:
print("\n=== ArgMax Baseline Predictions ===")
results_argmax = run_test_with_thresholds(
    model, tokenizer, test_df, label_encoders,
    loss_functions, train_config, '_argmax', thresholds=None
)